# initialiastion du projet de detection de la pneumonie 

In [ ]:
import tensorflow as tf # Bibliothèque principale pour la création et l'entraînement de modèles de Deep Learning
from tensorflow.keras import layers, models # Importation des outils pour construire les couches (Conv2D, Dense) et la structure (Sequential)
from tensorflow.keras.utils import plot_model # Outil pour générer un schéma visuel de l'architecture de votre réseau de neurones
import matplotlib.pyplot as plt # Bibliothèque pour tracer les graphiques (Accuracy, Loss) et afficher les images médicales
import numpy as np # Outil de calcul numérique pour manipuler les données (tableaux d'images et de labels)
from sklearn.metrics import classification_report # Outil pour calculer la Précision, le Rappel (Recall) et le F1-Score
from sklearn.metrics import confusion_matrix # Importation spécifique pour créer la matrice de confusion
import seaborn as sns # Bibliothèque de visualisation pour rendre la matrice de confusion plus lisible et esthétiques
import os # Bibliothèque pour interagir avec le système d'exploitation (gestion des dossiers et des fichiers d'images)


## Fonction pour afficher un certain nombre d’images d’un dossier

In [ ]:
# Import des bibliothèques nécessaires
import matplotlib.pyplot as plt
import os
from PIL import Image

# Fonction pour afficher un certain nombre d’images d’un dossier
def afficher_images(directory, class_name, n=4):
    print(f"\n--- {n} images pour {class_name} class ---")
    images_to_show = []
    # Parcours des fichiers dans le dossier
    for filename in os.listdir(directory):

        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            images_to_show.append(os.path.join(directory, filename))
        if len(images_to_show) == n:
            break

    plt.figure(figsize=(10, 5))
    for i, img_path in enumerate(images_to_show):
        img = Image.open(img_path)
        plt.subplot(1, n, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"{class_name} {i+1}")
        plt.axis('off')

    # Affichage final
    plt.show()


## Analyse des classes

In [ ]:
# Analyse des classes (Comptage du nombre d’images par classe)
Nbr_Nor = len(os.listdir(chemin+"/chest_xray/train/NORMAL"))
Nbr_Pneu= len(os.listdir(chemin+"/chest_xray/train/PNEUMONIA"))
print("Images NORMAL :", Nbr_Nor)
print("Images PNEUMONIA :", Nbr_Pneu)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


classes = ['NORMAL', 'PNEUMONIA']
counts = [Nbr_Nor, Nbr_Pneu]

plt.figure(figsize=(8, 6))
plt.bar(classes, counts, color=['skyblue', 'lightcoral'])
plt.xlabel('Classes')
plt.ylabel("Nombre d'images")
plt.title("Distribution des images par classe avant augmentation")
plt.show()

In [ ]:
# taille des images de la classe NORMAL soucre
from PIL import Image #

normal_dir = chemin + "/chest_xray/train/NORMAL"

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

In [ ]:
# taille des images de la classe NORMAL soucre
from PIL import Image #

normal_dir = chemin + "/chest_xray/train/NORMAL"

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

# Generation methode de Rotation(ImageDataGenerator de tensorflow)

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

normal_dir = chemin + "/chest_xray/train/NORMAL"
extensions = ('.jpg', '.jpeg', '.png')

# Nombre actuel d’images
current_count = len([f for f in os.listdir(normal_dir) if f.lower().endswith(extensions)])

target = 3875 # Objectif de 3875 images pour la classe NORMAL (équilibrage avec la classe PNEUMONIA)
generated = 0

original_images = [f for f in os.listdir(normal_dir) if f.lower().endswith(extensions)]

# Définir un répertoire accessible en écriture pour les images augmentées
save_augmented_dir = '/content/normal_rotation_augmente'
os.makedirs(save_augmented_dir, exist_ok=True)

for img_name in original_images:

    if current_count + generated >= target:
        break

    img_path = os.path.join(normal_dir, img_name)
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = x.reshape((1,) + x.shape)

    for batch in datagen.flow(
            x,
            batch_size=1,
            save_to_dir=save_augmented_dir, # Modifié en répertoire accessible en écriture
            save_prefix='aug',
            save_format='jpeg'):

        generated += 1

        if current_count + generated >= target:
            break

print("Total NORMAL :", current_count + generated)

## nombres d'image generer

In [ ]:
# Verication du nombre d'image generer
Nbr_Nor = len(os.listdir("/content/normal_rotation_augmente"))
print("Images NORMAL :", Nbr_Nor)

## taille des images de la classe NORMAL augmentee


In [ ]:
# taille des images de la classe NORMAL augmentee
from PIL import Image #

normal_dir = "/content/normal_rotation_augmente" # chemin en chemin vers les images augmentee

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

## Affichage des images de la classe Normal du Data Augmentation

In [ ]:
# Affichage de 5 images de la classe NORMAL augmentee
train_dir = "/content/normal_rotation_augmente"
# affichage des images de la classe NORMAL
afficher_images(train_dir, "NORMAL ROT",5)

### Étape 1 du GAN : Chargement des images et prétraitement

In [ ]:
# Étape 1 : Chargement des images et prétraitement
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

IMG_SIZE = 224  # taille des images
LATENT_DIM = 100  # dimension du vecteur latent
BATCH_SIZE = 32  # taille du batch
EPOCHS = 1000  # nombre d’époques

normal_dir = chemin + "/chest_xray/train/NORMAL"  # dossier NORMAL
target = 3875  # objectif d’images
images = []  # stockage des images

# parcourir les fichiers du dossier
for file in os.listdir(normal_dir):

    # vérifier que le fichier est une image
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = load_img(os.path.join(normal_dir, file), target_size=(IMG_SIZE, IMG_SIZE), color_mode="grayscale")  # chargement + resize + grayscale
        img = img_to_array(img)  # conversion numpy
        images.append(img)  # ajout à la liste

images = np.array(images, dtype=np.float32)  # conversion en array

# vérifier qu’il y a des images
if len(images) == 0:
    raise ValueError(f"Aucune image trouvée dans {normal_dir}")

images = (images - 127.5) / 127.5  # normalisation [-1,1]

### Étape 2 du GAN — Générateur (sortie : 224x224x1)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# Étape 2 — Générateur (sortie : 224x224x1)
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(LATENT_DIM,)),  # vecteur latent en entrée

        layers.Dense(7 * 7 * 256, use_bias=False),  # projection dense initiale
        layers.BatchNormalization(),  # stabilisation
        layers.LeakyReLU(),  # activation

        layers.Reshape((7, 7, 256)),  # reshape en feature map

        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),  # 14x14
        layers.BatchNormalization(),  # normalisation
        layers.LeakyReLU(),  # activation

        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),  # 28x28
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(32, 5, strides=2, padding="same", use_bias=False),  # 56x56
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(16, 5, strides=2, padding="same", use_bias=False),  # 112x112
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, 5, strides=2, padding="same", use_bias=False, activation="tanh")  # 224x224x1
    ])
    return model

### Étape 3 du GAN — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)

In [ ]:
# Étape 3 — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),  # image en entrée

        layers.Conv2D(32, 5, strides=2, padding="same"),  # réduction taille
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Conv2D(128, 5, strides=2, padding="same"),  # extraction features
        layers.BatchNormalization(),  # stabilisation (optionnel)
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, 5, strides=2, padding="same"),  # features profondes
        layers.BatchNormalization(),  # stabilisation (optionnel)
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),  # mise à plat
        layers.Dense(1)  # sortie (réel/faux)
    ])
    return model

### Étape 4 du GAN — Compilation

In [ ]:
# Étape 4 — Compilation
generator = build_generator()  # création du générateur
discriminator = build_discriminator()  # création du discriminateur

cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)  # fonction de perte

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)  # le générateur veut tromper le discriminateur

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)  # perte sur vraies images avec label smoothing
    # real_loss = cross_entropy(tf.ones_like(real_output), real_output)  # version sans label smoothing
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)  # perte sur fausses images
    return real_loss + fake_loss  # perte totale du discriminateur

gen_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur du générateur
disc_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur du discriminateur

### Étape 5 du GAN — Boucle d'entraînement

In [ ]:
# Étape 5 — Boucle d'entraînement
dataset = (
    tf.data.Dataset.from_tensor_slices(images)  # création dataset
    .shuffle(len(images))  # mélange
    .batch(BATCH_SIZE)  # batch (taille variable)
    .prefetch(tf.data.AUTOTUNE)  # optimisation
)

@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]  # taille batch dynamique
    noise = tf.random.normal([batch_size, LATENT_DIM])  # bruit aléatoire

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)  # génération images

        real_output = discriminator(real_images, training=True)  # prédiction réel
        fake_output = discriminator(generated_images, training=True)  # prédiction fake

        gen_loss = generator_loss(fake_output)  # perte générateur
        disc_loss = discriminator_loss(real_output, fake_output)  # perte discriminateur

    gradients_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)  # gradients G
    gradients_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)  # gradients D

    gen_optimizer.apply_gradients(zip(gradients_gen, generator.trainable_variables))  # update G
    disc_optimizer.apply_gradients(zip(gradients_disc, discriminator.trainable_variables))  # update D

    return gen_loss, disc_loss


# boucle sur les époques
for epoch in range(1, EPOCHS + 1):
    g_losses, d_losses = [], []  # stockage pertes

    # boucle sur les batches
    for image_batch in dataset:
        g_loss, d_loss = train_step(image_batch)
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    # afficher certaines époques
    if epoch % 1000 == 0 or epoch == 800 or epoch == 500 or epoch == 300 or epoch == 1:
        print(
            f"Epoch {epoch}/{EPOCHS} - "
            f"G_loss: {tf.reduce_mean(g_losses):.4f} - "
            f"D_loss: {tf.reduce_mean(d_losses):.4f}"
        )

### Étape 6 du GAN — Générer les images manquantes

In [ ]:
# Étape 6 — Générer les images manquantes
current_count = len(images)  # nombre actuel d’images
to_generate = max(0, target - current_count)  # nombre à générer

# vérifier s’il faut générer des images
if to_generate == 0:
    print(f"Rien à générer. Images actuelles: {current_count}, cible: {target}")
else:
    noise = tf.random.normal([to_generate, LATENT_DIM])  # bruit aléatoire
    generated_images = generator(noise, training=False)  # génération images

    generated_images = ((generated_images + 1.0) * 127.5).numpy()  # retour [0,255]
    generated_images = np.clip(generated_images, 0, 255).astype(np.uint8)  # nettoyage valeurs

    gan_output_dir = '/content/normal_gan_augmented'  # dossier de sortie
    os.makedirs(gan_output_dir, exist_ok=True)  # création dossier

    # sauvegarder chaque image générée
    for i in range(to_generate):
        out_path = os.path.join(gan_output_dir, f"normal_gan_{i:05d}.png")
        save_img(out_path, generated_images[i], scale=False)  # sauvegarde image

    print("Nouvelles images générées pour la classe NORMAL :", to_generate)

### taille des images de la classe NORMAL augmentée par GAN

In [ ]:
# taille des images de la classe NORMAL augmentée par GAN
from PIL import Image  # gestion des images

normal_dir = "/content/normal_gan_augmented"  # dossier images générées
img_name = os.listdir(normal_dir)[0]  # première image du dossier
img_path = os.path.join(normal_dir, img_name)  # chemin complet
img = Image.open(img_path)  # ouverture image

print("Nom image NORMAL:", img_name)  # affichage nom
print("Taille image NORMAL (largeur, hauteur) :", img.size)  # affichage dimensions

In [ ]:
# Verication du nombre d'image generer
Nbr_Nor = len(os.listdir("/content/normal_gan_augmented"))
print("Nombre d'images de la classe NORMAL generer :", Nbr_Nor)

In [ ]:
# Affichage de 5 images de la classe NORMAL augmentée par GAN
train_dir = "/content/normal_gan_augmented"

# affichage des images de la classe NORMAL

afficher_images(train_dir, "NORMAL GAN",5)

## <center> Augmentation par la methode GAN des images de la classes PNEUMONIA pour atteindre 6417 images</center>

### Etape 1 GAN2: Chargement images PNEUMONIA

In [ ]:
# imports
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# paramètres
IMG_SIZE = 224  # taille images
LATENT_DIM = 100  # vecteur latent
BATCH_SIZE = 32  # batch size
EPOCHS = 1000  # époques

pneumonia_dir = chemin + "/chest_xray/train/PNEUMONIA"  # dossier PNEUMONIA
target = 6417  # objectif images

images = []  # stockage images

# parcourir les fichiers
for file in os.listdir(pneumonia_dir):

    # vérifier que c’est une image
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = load_img(os.path.join(pneumonia_dir, file), target_size=(IMG_SIZE, IMG_SIZE), color_mode="grayscale")  # chargement + resize
        img = img_to_array(img)  # conversion numpy
        images.append(img)  # ajout liste

images = np.array(images, dtype=np.float32)  # conversion array

# vérifier présence images
if len(images) == 0:
    raise ValueError(f"Aucune image trouvée dans {pneumonia_dir}")

images = (images - 127.5) / 127.5  # normalisation [-1,1]

### Étape 2 GAN2 — Générateur

In [ ]:
# imports
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# Étape 2 — Générateur (sortie : 224x224x1)
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(LATENT_DIM,)),  # entrée bruit

        layers.Dense(7 * 7 * 256, use_bias=False),  # projection initiale
        layers.BatchNormalization(),  # normalisation
        layers.LeakyReLU(),  # activation

        layers.Reshape((7, 7, 256)),  # reshape feature map

        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),  # 14x14
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),  # 28x28
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(32, 5, strides=2, padding="same", use_bias=False),  # 56x56
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(16, 5, strides=2, padding="same", use_bias=False),  # 112x112
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, 5, strides=2, padding="same", use_bias=False, activation="tanh")  # 224x224x1
    ])
    return model

### Étape 3 GAN2 — Discriminateur

## taille des images de la classe PNEUMONIA augmentée par GAN2

In [ ]:
# imports
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# Étape 3 — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),  # image en entrée
        layers.Conv2D(32, 5, strides=2, padding="same"),  # extraction features
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Conv2D(128, 5, strides=2, padding="same"),  # extraction profonde
        layers.BatchNormalization(),  # stabilisation
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Conv2D(256, 5, strides=2, padding="same"),  # features avancées
        layers.BatchNormalization(),  # stabilisation
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Flatten(),  # mise à plat
        layers.Dense(1)  # sortie finale
    ])
    return model

### Étape 4 GAN2 — Compilation

In [ ]:
# Étape 4 — Compilation
generator = build_generator()  # init générateur
discriminator = build_discriminator()  # init discriminateur

cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)  # loss binaire

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)  # objectif tromper D

def discriminator_loss(real_output, fake_output):
    # utiliser label smoothing (0.9 au lieu de 1)
    real_loss = cross_entropy(tf.ones_like(real_output)*0.9, real_output)
    # real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)  # faux = 0
    return real_loss + fake_loss  # perte totale

gen_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur G
disc_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur D


### Étape 5 GAN2 — Boucle d'entraînement

In [ ]:
  # Étape 5 — Boucle d'entraînement
dataset = (
    tf.data.Dataset.from_tensor_slices(images)  # créer dataset
    .shuffle(len(images))  # mélanger les images
    .batch(BATCH_SIZE)  # créer les batches
    .prefetch(tf.data.AUTOTUNE)  # optimiser le chargement
)

@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]  # taille dynamique du batch
    noise = tf.random.normal([batch_size, LATENT_DIM])  # générer le bruit

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)  # générer des images
        real_output = discriminator(real_images, training=True)  # sortie vraies images
        fake_output = discriminator(generated_images, training=True)  # sortie fausses images
        gen_loss = generator_loss(fake_output)  # perte générateur
        disc_loss = discriminator_loss(real_output, fake_output)  # perte discriminateur

    gradients_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)  # gradients générateur
    gradients_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)  # gradients discriminateur
    gen_optimizer.apply_gradients(zip(gradients_gen, generator.trainable_variables))  # mise à jour générateur
    disc_optimizer.apply_gradients(zip(gradients_disc, discriminator.trainable_variables))  # mise à jour discriminateur

    return gen_loss, disc_loss

# parcourir les époques
for epoch in range(1, EPOCHS + 1):
    g_losses, d_losses = [], []  # stocker les pertes

    # parcourir les batches
    for image_batch in dataset:
        g_loss, d_loss = train_step(image_batch)
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    # afficher les pertes à certaines époques
    if epoch % 1000 == 0 or epoch == 800 or epoch == 500 or epoch == 300 or epoch == 1:
        print(
            f"Epoch {epoch}/{EPOCHS} - "
            f"G_loss: {tf.reduce_mean(g_losses):.4f} - "
            f"D_loss: {tf.reduce_mean(d_losses):.4f}"
        )

### Étape 6 GAN2 — Générer les images manquantes de PNEUMONIA

In [ ]:
# Étape 6 — Générer les images manquantes de PNEUMONIA
current_count = len(images)  # nombre actuel
to_generate = max(0, target - current_count)  # nombre à générer

# vérifier s’il faut générer
if to_generate == 0:
    print(f"Rien à générer. Images actuelles: {current_count}, cible: {target}")
else:
    noise = tf.random.normal([to_generate, LATENT_DIM])  # bruit aléatoire
    generated_images = generator(noise, training=False)  # génération images

    generated_images = ((generated_images + 1.0) * 127.5).numpy()  # [-1,1] -> [0,255]
    generated_images = np.clip(generated_images, 0, 255).astype(np.uint8)  # nettoyage valeurs

    gan_output_dir = '/content/pneumonia_gan_augmented'  # dossier sortie
    os.makedirs(gan_output_dir, exist_ok=True)  # créer dossier

    # sauvegarder les images
    for i in range(to_generate):
        out_path = os.path.join(gan_output_dir, f"pneum_gan_{i:05d}.png")
        save_img(out_path, generated_images[i], scale=False)  # sauvegarde

    print("Nouvelles images générées pour la classe PNEUMONIA :", to_generate)

## taille des images de la classe PNEUMONIA augmentée par GAN2

In [ ]:
# taille des images de la classe PNEUMONIA augmentée par GAN
from PIL import Image  # gestion images

normal_dir = "/content/pneumonia_gan_augmented"  # dossier images GAN

img_name = os.listdir(normal_dir)[0]  # première image
img_path = os.path.join(normal_dir, img_name)  # chemin complet

img = Image.open(img_path)  # ouvrir image

print("Nom image PNEUMONIA:", img_name)  # afficher nom
print("Taille image PNEUMONIA GAN (largeur, hauteur) :", img.size)  # afficher taille

In [ ]:
# nombre d'image generer pour la classe PNEUMONIA
Nbr_Nor = len(os.listdir("/content/pneumonia_gan_augmented"))
print("Nombre d'images de la classe PNEUMONIA generer :", Nbr_Nor)

In [ ]:
# Affichage de 5 images de la classe PNEUMONIA augmentée par GAN
train_dir = "/content/pneumonia_gan_augmented"

# affichage des images de la classe PNEUMONIA

afficher_images(train_dir, "PNEUMONIA GAN",5)

## Redimensionner et Sauvegarder les Images NORMAL Originales<br>Créer le répertoire `/content/normal_224`, parcourir les images NORMAL originales, les redimensionner à 224x224 et les sauvegarder dans ce nouveau répertoire.


### Redimensionnement images NORMAL (224x224)

In [ ]:
import os
from PIL import Image  # gestion images

original_normal_dir = chemin + "/chest_xray/train/NORMAL"  # dossier source
resized_normal_dir = '/content/normal_224'  # dossier destination

os.makedirs(resized_normal_dir, exist_ok=True)  # créer dossier si absent

IMG_SIZE = (224, 224)  # taille cible
processed_count = 0  # compteur

print(f"Redimensionnement des images de {original_normal_dir} vers {resized_normal_dir}")

# parcourir les fichiers
for filename in os.listdir(original_normal_dir):

    # vérifier que c’est une image
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path_source = os.path.join(original_normal_dir, filename)  # chemin source
        img_path_destination = os.path.join(resized_normal_dir, filename)  # chemin destination

        # gérer erreurs lecture image
        try:
            img = Image.open(img_path_source)  # ouvrir image
            img_resized = img.resize(IMG_SIZE)  # redimensionner
            img_resized.save(img_path_destination)  # sauvegarder
            processed_count += 1  # incrémenter compteur
        except Exception as e:
            print(f"Erreur lors du traitement de l'image {filename}: {e}")  # afficher erreur

print(f"Terminé. {processed_count} images NORMAL redimensionnées et sauvegardées dans {resized_normal_dir}.")

In [ ]:
# nombre d'images originales de la classe NORMAL redimensionnées
Nbr_Nor = len(os.listdir("/content/normal_224"))
print("Images NORMAL :", Nbr_Nor)

In [ ]:
# taille des images dans le dossier normal_224
from PIL import Image #

normal_dir = "/content/normal_224" # chemin vers le dossier normal_224

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

## Redimensionner et Sauvegarder les Images PNEUMONIA Originales<br>Créer le répertoire `/content/pneumonia_224`, parcourir les images PNEUMONIA originales, les redimensionner à 224x224 et les sauvegarder dans ce nouveau répertoire.


### Redimensionnement images PNEUMONIA (224x224)

In [ ]:
import os
from PIL import Image  # gestion images

original_pneumonia_dir = chemin + "/chest_xray/train/PNEUMONIA"  # dossier source
resized_pneumonia_dir = '/content/pneumonia_224'  # dossier destination

os.makedirs(resized_pneumonia_dir, exist_ok=True)  # créer dossier

IMG_SIZE = (224, 224)  # taille cible
processed_count = 0  # compteur

print(f"Redimensionnement des images de {original_pneumonia_dir} vers {resized_pneumonia_dir}")

# parcourir les fichiers
for filename in os.listdir(original_pneumonia_dir):

    # vérifier que c’est une image
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path_source = os.path.join(original_pneumonia_dir, filename)  # chemin source
        img_path_destination = os.path.join(resized_pneumonia_dir, filename)  # chemin destination

        # gérer erreurs traitement image
        try:
            img = Image.open(img_path_source)  # ouvrir image
            img_resized = img.resize(IMG_SIZE)  # redimensionner
            img_resized.save(img_path_destination)  # sauvegarder
            processed_count += 1  # incrémenter compteur
        except Exception as e:
            print(f"Erreur lors du traitement de l'image {filename}: {e}")  # afficher erreur

print(f"Terminé. {processed_count} images PNEUMONIA redimensionnées et sauvegardées dans {resized_pneumonia_dir}.")

In [ ]:
# Nombre d'images originales de la classe PNEUMONIA redimensionnées
Nbr_Pneu_resized = len(os.listdir("/content/pneumonia_224"))
print("nombre d'images PNEUMONIA redimensionnées :", Nbr_Pneu_resized)


In [ ]:
# taille des images dans le dossier pneumonia_224
from PIL import Image #

normal_dir = "/content/pneumonia_224" # chemin vers le dossier pneumonia_224

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

## creation du jeux de donnee final fusionné

## <center> Fussion des images GAN,DATA AUGMENTATION et le classe minoritaire dans un seul dossier(dossier NORMAL et dossier PNEUMONIA) Dans google drive </center>

In [ ]:
import os
import shutil

chemin_sortie = "/content/drive/MyDrive/Mes_Projet/Projet"  # dossier sortie
merged_base_dir = chemin_sortie + '/Dataset_E'  # dataset final

merged_normal_dir = os.path.join(merged_base_dir, 'NORMAL')  # dossier NORMAL
merged_pneumonia_dir = os.path.join(merged_base_dir, 'PNEUMONIA')  # dossier PNEUMONIA

os.makedirs(merged_normal_dir, exist_ok=True)  # créer NORMAL
os.makedirs(merged_pneumonia_dir, exist_ok=True)  # créer PNEUMONIA
print(f"Dossiers créés : {merged_normal_dir} et {merged_pneumonia_dir}")

# sources NORMAL
normal_rotation_aug_dir = '/content/normal_rotation_augmente'
normal_gan_aug_dir = '/content/normal_gan_augmented'
normal_original_resized_dir = '/content/normal_224'

# sources PNEUMONIA
pneumonia_gan_aug_dir = '/content/pneumonia_gan_augmented'
pneumonia_original_resized_dir = '/content/pneumonia_224'

# copier images avec gestion doublons
def copy_images(source_dir, destination_dir):
    count = 0  # compteur

    # vérifier existence dossier source
    if os.path.exists(source_dir):

        # parcourir fichiers
        for filename in os.listdir(source_dir):

            # vérifier que c’est une image
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                source_path = os.path.join(source_dir, filename)  # chemin source
                destination_path = os.path.join(destination_dir, filename)  # chemin destination

                base_name, extension = os.path.splitext(filename)  # séparer nom/ext
                counter = 1  # compteur renommage

                # gérer doublons
                while os.path.exists(destination_path):
                    new_filename = f"{base_name}_{counter}{extension}"
                    destination_path = os.path.join(destination_dir, new_filename)
                    counter += 1

                shutil.copy(source_path, destination_path)  # copier fichier
                count += 1  # incrémenter

    return count


print("\n--- Copie des images NORMAL ---")
count_normal_rot = copy_images(normal_rotation_aug_dir, merged_normal_dir)  # rotation
print(f"{count_normal_rot} images copiées depuis {normal_rotation_aug_dir}")

count_normal_gan = copy_images(normal_gan_aug_dir, merged_normal_dir)  # GAN
print(f"{count_normal_gan} images copiées depuis {normal_gan_aug_dir}")

count_normal_orig = copy_images(normal_original_resized_dir, merged_normal_dir)  # originales
print(f"{count_normal_orig} images copiées depuis {normal_original_resized_dir}")


print("\n--- Copie des images PNEUMONIA ---")
count_pneumonia_gan = copy_images(pneumonia_gan_aug_dir, merged_pneumonia_dir)  # GAN
print(f"{count_pneumonia_gan} images copiées depuis {pneumonia_gan_aug_dir}")

count_pneumonia_orig = copy_images(pneumonia_original_resized_dir, merged_pneumonia_dir)  # originales
print(f"{count_pneumonia_orig} images copiées depuis {pneumonia_original_resized_dir}")


total_normal_images = len(os.listdir(merged_normal_dir))  # total NORMAL
total_pneumonia_images = len(os.listdir(merged_pneumonia_dir))  # total PNEUMONIA

print(f"\nTotal d'images dans {merged_normal_dir}: {total_normal_images}")
print(f"Total d'images dans {merged_pneumonia_dir}: {total_pneumonia_images}")

## Création du dans google drive du dossier Dataset_D (qui stocke les images originales uniquement)

In [ ]:
import os
import shutil

chemin_sortie = "/content/drive/MyDrive/Mes_Projet/Projet"  # dossier sortie
merged_base_dir_D = chemin_sortie + '/Dataset_D'  # dataset D

merged_normal_dir_D = os.path.join(merged_base_dir_D, 'NORMAL')  # dossier NORMAL
merged_pneumonia_dir_D = os.path.join(merged_base_dir_D, 'PNEUMONIA')  # dossier PNEUMONIA

os.makedirs(merged_normal_dir_D, exist_ok=True)  # créer NORMAL
os.makedirs(merged_pneumonia_dir_D, exist_ok=True)  # créer PNEUMONIA
print(f"Dossiers créés : {merged_normal_dir_D} et {merged_pneumonia_dir_D}")

normal_original_resized_dir = '/content/normal_224'  # source NORMAL
pneumonia_original_resized_dir = '/content/pneumonia_224'  # source PNEUMONIA

# copier images avec gestion doublons
def copy_images(source_dir, destination_dir):
    count = 0  # compteur

    # vérifier existence dossier source
    if os.path.exists(source_dir):

        # parcourir fichiers
        for filename in os.listdir(source_dir):

            # vérifier que c’est une image
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                source_path = os.path.join(source_dir, filename)  # chemin source
                destination_path = os.path.join(destination_dir, filename)  # chemin destination

                base_name, extension = os.path.splitext(filename)  # séparer nom/ext
                counter = 1  # compteur renommage

                # gérer doublons
                while os.path.exists(destination_path):
                    new_filename = f"{base_name}_{counter}{extension}"
                    destination_path = os.path.join(destination_dir, new_filename)
                    counter += 1

                shutil.copy(source_path, destination_path)  # copier
                count += 1  # incrémenter

    return count


print("\n--- Copie des images NORMAL ---")
count_normal_D = copy_images(normal_original_resized_dir, merged_normal_dir_D)  # copie NORMAL
print(f"{count_normal_D} images copiées depuis {normal_original_resized_dir} vers {merged_normal_dir_D}")


print("\n--- Copie des images PNEUMONIA ---")
count_pneumonia_D = copy_images(pneumonia_original_resized_dir, merged_pneumonia_dir_D)  # copie PNEUMONIA
print(f"{count_pneumonia_D} images copiées depuis {pneumonia_original_resized_dir} vers {merged_pneumonia_dir_D}")


total_normal_images_D = len(os.listdir(merged_normal_dir_D))  # total NORMAL
total_pneumonia_images_D = len(os.listdir(merged_pneumonia_dir_D))  # total PNEUMONIA

print(f"\nTotal d'images dans {merged_normal_dir_D}: {total_normal_images_D}")
print(f"Total d'images dans {merged_pneumonia_dir_D}: {total_pneumonia_images_D}")

## Histogramme des couleurs avant contraste

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# chemin vers ton dataset
dataset_path = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E/PNEUMONIA"

# liste des images
images = os.listdir(dataset_path)[10:14]

plt.figure(figsize=(12,8))

for i, img_name in enumerate(images):

    img_path = os.path.join(dataset_path, img_name)

    image = cv2.imread(img_path, 0)  # lecture en niveaux de gris

    plt.subplot(2,2,i+1)
    plt.hist(image.ravel(), 256, [0,256])
    plt.title(f"Histogramme {img_name}")

plt.tight_layout()
plt.show()

# contraster les images d'entrainement du dataset equilibrer (Dataset_E)

In [ ]:
import os
import cv2  # OpenCV

input_base_folder = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E"  # dossier source
output_base_folder = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E_CONT"  # dossier sortie

os.makedirs(output_base_folder, exist_ok=True)  # créer dossier sortie

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))  # initialiser CLAHE

print(f"Applying CLAHE to images from {input_base_folder} and saving to {output_base_folder}")

# parcourir les classes
for class_name in os.listdir(input_base_folder):
    input_class_path = os.path.join(input_base_folder, class_name)  # chemin entrée
    output_class_path = os.path.join(output_base_folder, class_name)  # chemin sortie

    # vérifier que c’est un dossier
    if os.path.isdir(input_class_path):
        os.makedirs(output_class_path, exist_ok=True)  # créer dossier classe
        print(f"Processing images in {input_class_path}...")

        # parcourir les images
        for img_name in os.listdir(input_class_path):

            # vérifier que c’est une image
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path_source = os.path.join(input_class_path, img_name)  # source
                img_path_destination = os.path.join(output_class_path, img_name)  # destination

                img = cv2.imread(img_path_source, cv2.IMREAD_GRAYSCALE)  # lire image

                # vérifier chargement image
                if img is not None:
                    enhanced = clahe.apply(img)  # appliquer CLAHE
                    cv2.imwrite(img_path_destination, enhanced)  # sauvegarder
                else:
                    print(f"Warning: Could not read image {img_path_source}. Skipping.")
    else:
        print(f"Warning: Skipping non-directory item in {input_base_folder}: {class_name}")

print("CLAHE application complete.")

# Preparation des fichiers test

In [ ]:
# Vérification du contenu du dossier téléchargé
print(os.listdir(chemin +"/chest_xray/test")) # Affiche les fichiers et dossiers présents dans le chemin spécifié

### Redimensionnement images TEST (NORMAL + PNEUMONIA)

In [ ]:
import os
from PIL import Image  # gestion images

original_test_normal_dir = chemin + "/chest_xray/test/NORMAL"  # source NORMAL
original_test_pneumonia_dir = chemin + "/chest_xray/test/PNEUMONIA"  # source PNEUMONIA

resized_test_normal_dir = '/content/test_normal_224'  # destination NORMAL
resized_test_pneumonia_dir = '/content/test_pneumonia_224'  # destination PNEUMONIA

os.makedirs(resized_test_normal_dir, exist_ok=True)  # créer NORMAL
os.makedirs(resized_test_pneumonia_dir, exist_ok=True)  # créer PNEUMONIA

IMG_SIZE = (224, 224)  # taille cible

# redimensionner et sauvegarder images
def resize_and_save_images(source_dir, destination_dir, class_name):
    processed_count = 0  # compteur
    print(f"\nRedimensionnement des images de {source_dir} vers {destination_dir}")

    # parcourir les fichiers
    for filename in os.listdir(source_dir):

        # vérifier que c’est une image
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path_source = os.path.join(source_dir, filename)  # chemin source
            img_path_destination = os.path.join(destination_dir, filename)  # chemin destination

            # gérer erreurs traitement
            try:
                img = Image.open(img_path_source)  # ouvrir image
                img_resized = img.resize(IMG_SIZE)  # redimensionner
                img_resized.save(img_path_destination)  # sauvegarder
                processed_count += 1  # incrémenter
            except Exception as e:
                print(f"Erreur lors du traitement de l'image {filename}: {e}")  # erreur

    print(f"Terminé. {processed_count} images {class_name} redimensionnées et sauvegardées dans {destination_dir}.")
    return processed_count


count_normal_test = resize_and_save_images(original_test_normal_dir, resized_test_normal_dir, "NORMAL")  # NORMAL
count_pneumonia_test = resize_and_save_images(original_test_pneumonia_dir, resized_test_pneumonia_dir, "PNEUMONIA")  # PNEUMONIA

print(f"\nTotal d'images NORMAL de test redimensionnées: {count_normal_test}")
print(f"Total d'images PNEUMONIA de test redimensionnées: {count_pneumonia_test}")

## Contraste des images de test pour augmenter le niveau de detail

In [ ]:
# contraste
import os
import cv2  # OpenCV

input_base_folder = "/content/drive/MyDrive/Mes_Projet/Projet/data_test"  # dossier source
output_base_folder = "/content/drive/MyDrive/Mes_Projet/Projet/data_test_CONT"  # dossier sortie

os.makedirs(output_base_folder, exist_ok=True)  # créer dossier sortie

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))  # initialiser CLAHE

print(f"Applying CLAHE to images from {input_base_folder} and saving to {output_base_folder}")

# parcourir les sous-dossiers de classes
for class_name in os.listdir(input_base_folder):
    input_class_path = os.path.join(input_base_folder, class_name)  # chemin dossier source
    output_class_path = os.path.join(output_base_folder, class_name)  # chemin dossier sortie

    # vérifier que c’est un dossier
    if os.path.isdir(input_class_path):
        os.makedirs(output_class_path, exist_ok=True)  # créer dossier classe
        print(f"Processing images in {input_class_path}...")

        # parcourir les images du dossier
        for img_name in os.listdir(input_class_path):

            # vérifier que c’est une image
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path_source = os.path.join(input_class_path, img_name)  # chemin image source
                img_path_destination = os.path.join(output_class_path, img_name)  # chemin image sortie

                img = cv2.imread(img_path_source, cv2.IMREAD_GRAYSCALE)  # lire en niveau de gris

                # vérifier que l’image est bien chargée
                if img is not None:
                    enhanced = clahe.apply(img)  # appliquer CLAHE
                    cv2.imwrite(img_path_destination, enhanced)  # sauvegarder image
                else:
                    print(f"Warning: Could not read image {img_path_source}. Skipping.")
    else:
        print(f"Warning: Skipping non-directory item in {input_base_folder}: {class_name}")

print("CLAHE application complete.")

## Histogramme des couleurs apres contraste

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# chemin vers ton dataset
dataset_path = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E_CONT/PNEUMONIA"

# liste des images
images = os.listdir(dataset_path)[:4]

plt.figure(figsize=(12,8))

for i, img_name in enumerate(images):

    img_path = os.path.join(dataset_path, img_name)

    image = cv2.imread(img_path, 0)  # lecture en niveaux de gris

    plt.subplot(2,2,i+1)
    plt.hist(image.ravel(), 256, [0,256])
    plt.title(f"Histogramme {img_name}")

plt.tight_layout()
plt.show()

# Création du dataset de test final

In [ ]:
import os
import shutil

chemin_sortie = "/content/drive/MyDrive/Mes_Projet/Projet"  # dossier sortie
data_test_base_dir = os.path.join(chemin_sortie, 'data_test')  # dossier test

data_test_normal_dir = os.path.join(data_test_base_dir, 'NORMAL')  # NORMAL
data_test_pneumonia_dir = os.path.join(data_test_base_dir, 'PNEUMONIA')  # PNEUMONIA

os.makedirs(data_test_normal_dir, exist_ok=True)  # créer NORMAL
os.makedirs(data_test_pneumonia_dir, exist_ok=True)  # créer PNEUMONIA
print(f"Dossiers créés : {data_test_normal_dir} et {data_test_pneumonia_dir}")

resized_test_normal_src = '/content/test_normal_224'  # source NORMAL
resized_test_pneumonia_src = '/content/test_pneumonia_224'  # source PNEUMONIA

# copier images avec gestion doublons
def copy_images(source_dir, destination_dir):
    count = 0  # compteur

    # vérifier existence dossier source
    if os.path.exists(source_dir):

        # parcourir fichiers
        for filename in os.listdir(source_dir):

            # vérifier que c’est une image
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                source_path = os.path.join(source_dir, filename)  # source
                destination_path = os.path.join(destination_dir, filename)  # destination

                base_name, extension = os.path.splitext(filename)  # nom/ext
                counter = 1  # compteur renommage

                # gérer doublons
                while os.path.exists(destination_path):
                    new_filename = f"{base_name}_{counter}{extension}"
                    destination_path = os.path.join(destination_dir, new_filename)
                    counter += 1

                shutil.copy(source_path, destination_path)  # copier
                count += 1  # incrémenter

    return count


print("\n--- Copie des images NORMAL de test ---")
count_normal_copied = copy_images(resized_test_normal_src, data_test_normal_dir)  # NORMAL
print(f"{count_normal_copied} images copiées depuis {resized_test_normal_src} vers {data_test_normal_dir}")


print("\n--- Copie des images PNEUMONIA de test ---")
count_pneumonia_copied = copy_images(resized_test_pneumonia_src, data_test_pneumonia_dir)  # PNEUMONIA
print(f"{count_pneumonia_copied} images copiées depuis {resized_test_pneumonia_src} vers {data_test_pneumonia_dir}")


total_normal_test = len(os.listdir(data_test_normal_dir))  # total NORMAL
total_pneumonia_test = len(os.listdir(data_test_pneumonia_dir))  # total PNEUMONIA

print(f"\nTotal d'images NORMAL dans {data_test_normal_dir}: {total_normal_test}")
print(f"Total d'images PNEUMONIA dans {data_test_pneumonia_dir}: {total_pneumonia_test}")

---
# Entrainement des differents models de classification
----

# 1-Model de classification des images par CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# chemins des datasets drive
train_data_dir = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E_CONT"  # dossier train
test_data_dir = "/content/drive/MyDrive/Mes_Projet/Projet/data_test_CONT"  # dossier test

# # chemins des datasets local
# train_data_dir = "Dataset_E_CONT"  # dossier train
# test_data_dir = "data_test_CONT"  # dossier test

IMG_HEIGHT = 224  # hauteur image
IMG_WIDTH = 224  # largeur image
BATCH_SIZE = 32  # taille batch
NUM_CLASSES = 2  # nombre de classes
EPOCHS = 100  # nombre d'époques

# générateur train avec augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,  # normalisation
    rotation_range=20,  # rotation
    width_shift_range=0.2,  # décalage horizontal
    height_shift_range=0.2,  # décalage vertical
    shear_range=0.2,  # cisaillement
    zoom_range=0.2,  # zoom
    horizontal_flip=True,  # retournement horizontal
    fill_mode='nearest'  # remplissage
)

# générateur test sans augmentation
test_datagen = ImageDataGenerator(rescale=1./255)  # normalisation seulement

print("chargement des données d'entraînement...")

# gérer les erreurs de chargement train
try:
    train_generator = train_datagen.flow_from_directory(
        train_data_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='binary'  # classification binaire
    )
except Exception as e:
    print(f"Erreur de chargement des données d'entraînement depuis {train_data_dir}: {e}")
    print("Veuillez vous assurer que le dossier existe et contient des sous-dossiers pour chaque classe (NORMAL_D, PNEUMONIA_D).")
    raise

print("chargement des données de test...")

# gérer les erreurs de chargement test
try:
    test_generator = test_datagen.flow_from_directory(
        test_data_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='binary',  # classification binaire
        shuffle=False  # garder l’ordre pour l’évaluation
    )
except Exception as e:
    print(f"Erreur de chargement des données de test depuis {test_data_dir}: {e}")
    print("Veuillez vous assurer que le dossier existe et contient des sous-dossiers pour chaque classe (e.g., NORMAL, PNEUMONIA).")
    raise

print("construction du modèle CNN...")  # construire le modèle CNN
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),  # 1re convolution
    layers.MaxPooling2D((2, 2)),                    # sous-échantillonnage
    layers.Conv2D(64, (3, 3), activation='relu'),   # 2e convolution
    layers.MaxPooling2D((2, 2)),                    # sous-échantillonnage
    layers.Conv2D(128, (3, 3), activation='relu'),  # 3e convolution
    layers.MaxPooling2D((2, 2)),                    # sous-échantillonnage
    layers.Flatten(),                               # mise à plat
    layers.Dense(512, activation='relu'),           # couche dense
    layers.Dropout(0.5),                            # régularisation
    layers.Dense(1, activation='sigmoid')           # sortie binaire
])

print("Compilation du modèle...")                   # compiler le modèle
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00025),  # Optimiseur Adam avec learning rate
    #optimizer='adam',                           # taux d’apprentissage
    loss='binary_crossentropy',                     # fonction de perte
    metrics=['accuracy']                            # métrique
)

print("\nRésumé du modèle:")                        # afficher le résumé
model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  # Arrêt automatique au meilleur modèle

print("\nEntraînement du modèle...")                # entraîner le modèle
history = model.fit(
    train_generator,
    steps_per_epoch=int(np.ceil(train_generator.samples / BATCH_SIZE)), # Corrected to use np.ceil
    epochs=EPOCHS,
    validation_data=test_generator,
    validation_steps=int(np.ceil(test_generator.samples / BATCH_SIZE)), # Corrected to use np.ceil
    callbacks=[early_stop],  # Activation du EarlyStopping
    verbose=1  # Affichage
)

print("\nÉvaluation du modèle...")                  # évaluer le modèle
loss, accuracy = model.evaluate(test_generator)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

plt.figure(figsize=(12, 4))                         # créer figure
plt.subplot(1, 2, 1)                                # sous-graphique accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')  # accuracy train
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')  # accuracy validation
plt.legend()
plt.title('Accuracy over Epochs')

plt.subplot(1, 2, 2)                                # sous-graphique loss
plt.plot(history.history['loss'], label='Training Loss')  # loss train
plt.plot(history.history['val_loss'], label='Validation Loss')  # loss validation
plt.legend()
plt.title('Loss over Epochs')
plt.show()                                          # afficher graphiques

print("\nGénération des prédictions pour l'évaluation...")  # générer prédictions
test_generator.reset()                              # réinitialiser générateur
predictions = model.predict(test_generator)         # prédictions probabilités
predicted_classes = (predictions > 0.42).astype(int).flatten()  # convertir en classes
true_classes = test_generator.classes               # vraies classes

print("\nClassification Report:")                   # afficher rapport
target_names = [name for name, idx in sorted(train_generator.class_indices.items(), key=lambda item: item[1])]  # noms classes
print(classification_report(true_classes, predicted_classes, target_names=target_names))

cm = confusion_matrix(true_classes, predicted_classes)  # matrice confusion
plt.figure(figsize=(8, 6))                          # créer figure matrice
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)  # heatmap
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()                                          # afficher matrice

# 2-Model de classification des images par CNN avec class_weights

In [ ]:
import tensorflow as tf                                      # Importation de TensorFlow
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Générateur d'images pour le prétraitement
from tensorflow.keras import layers, models                  # Importation des couches et modèles Keras
import os                                                    # Module pour interagir avec le système
import matplotlib.pyplot as plt                              # Bibliothèque pour les graphiques
import numpy as np                                           # Bibliothèque pour les calculs numériques
from sklearn.metrics import classification_report, confusion_matrix  # Métriques d'évaluation
from sklearn.utils.class_weight import compute_class_weight  # Calcul des poids de classes
import seaborn as sns                                        # Visualisation (matrice de confusion)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'                     # Activation de l'affichage complet des logs TensorFlow

# train_data_dir = "Dataset_E_CONT"                            # Chemin des données d'entraînement
# test_data_dir = "data_test_CONT"                             # Chemin des données de test

train_data_dir = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E_CONT"  # Chemin alternatif (Google Drive)
test_data_dir = "/content/drive/MyDrive/Mes_Projet/Projet/data_test_CONT"   # Chemin alternatif (Google Drive)

IMG_HEIGHT = 224                                             # Hauteur des images
IMG_WIDTH = 224                                              # Largeur des images
BATCH_SIZE = 32                                              # Taille des batches
NUM_CLASSES = 2                                              # Nombre de classes (NORMAL, PNEUMONIA)
EPOCHS = 100                                                 # Nombre d'époques

# Création du générateur de données d'entraînement avec augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalisation des pixels entre 0 et 1
    rotation_range=20,  # Rotation aléatoire des images
    width_shift_range=0.2,  # Décalage horizontal aléatoire
    height_shift_range=0.2,  # Décalage vertical aléatoire
    shear_range=0.2,  # Transformation de cisaillement
    zoom_range=0.2,  # Zoom aléatoire
    horizontal_flip=True,  # Retournement horizontal
    fill_mode='nearest'  # Méthode de remplissage après transformation
)

# Création du générateur de données de test sans augmentation, seulement normalisation
test_datagen = ImageDataGenerator(rescale=1./255)

# Chargement des données d'entraînement
print("Loading training data...")
try:
    # Chargement des images d'entraînement depuis le dossier
    train_generator = train_datagen.flow_from_directory(
        train_data_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),  # Redimensionnement des images
        batch_size=BATCH_SIZE,  # Taille des lots
        class_mode='binary'  # Classification binaire
    )
except Exception as e:
    # Affichage de l'erreur en cas de problème de chargement
    print(f"Error loading training data from {train_data_dir}: {e}")
    print("Please ensure the directory exists and contains subdirectories for each class (e.g., NORMAL_D, PNEUMONIA_D).")
    raise

# Chargement des données de test
print("Loading test data...")
try:
    # Chargement des images de test depuis le dossier
    test_generator = test_datagen.flow_from_directory(
        test_data_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),  # Redimensionnement des images
        batch_size=BATCH_SIZE,  # Taille des lots
        class_mode='binary',  # Classification binaire
        shuffle=False  # Conservation de l'ordre pour l'évaluation
    )
except Exception as e:
    # Affichage de l'erreur en cas de problème de chargement
    print(f"Error loading test data from {test_data_dir}: {e}")
    print("Please ensure the directory exists and contains subdirectories for each class (e.g., NORMAL, PNEUMONIA).")
    raise

# Récupération des étiquettes des données d'entraînement
labels = train_generator.classes

# Calcul des poids de classes pour corriger le déséquilibre entre NORMAL et PNEUMONIA
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

# Conversion des poids en dictionnaire pour les utiliser dans model.fit()
class_weights = dict(enumerate(class_weights))

# Affichage des poids calculés
print("Class weights:", class_weights)

# Construction du modèle CNN
print("Building CNN model...")
model = models.Sequential([

    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),# Première couche de convolution avec 32 filtres
    layers.MaxPooling2D((2, 2)),# Première couche de pooling pour réduire la dimension


    layers.Conv2D(64, (3, 3), activation='relu'),# Deuxième couche de convolution avec 64 filtres
    layers.MaxPooling2D((2, 2)),# Deuxième couche de pooling


    layers.Conv2D(128, (3, 3), activation='relu'),# Troisième couche de convolution avec 128 filtres
    layers.MaxPooling2D((2, 2)), # Troisième couche de pooling


    layers.Flatten(),# Aplatissement des cartes de caractéristiques en vecteur
    layers.Dense(512, activation='relu'),# Couche dense entièrement connectée avec 512 neurones
    layers.Dropout(0.5),# Dropout pour réduire le surapprentissage
    layers.Dense(1, activation='sigmoid')# Couche de sortie avec une seule sortie pour la classification binaire
])

# Compilation du modèle
print("Compiling model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00025),  #0.0001 Optimiseur Adam avec un taux d'apprentissage fixé
    loss='binary_crossentropy',  # Fonction de coût pour classification binaire
    metrics=['accuracy']  # Métrique d'évaluation
)

# Affichage du résumé du modèle
print("\nModel Summary:")
model.summary()

# Mise en place du EarlyStopping pour arrêter l'entraînement si la validation ne s'améliore plus
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # Surveillance de la perte sur les données de validation
    patience=10,  # Nombre d'époques à attendre avant arrêt
    restore_best_weights=True  # Restauration des meilleurs poids du modèle
)

# Entraînement du modèle
print("\nTraining model...")
history = model.fit(
    train_generator,
    steps_per_epoch=int(np.ceil(train_generator.samples / BATCH_SIZE)),  # Nombre de pas par époque
    epochs=EPOCHS,  # Nombre maximal d'époques
    validation_data=test_generator,  # Données de validation
    validation_steps=int(np.ceil(test_generator.samples / BATCH_SIZE)),  # Nombre de pas de validation
    callbacks=[early_stop],  # Activation du EarlyStopping
    class_weight=class_weights,  # Application des poids de classes
    verbose=1  # Affichage détaillé pendant l'entraînement
)

# Évaluation du modèle sur les données de test
print("\nEvaluating model...")
loss, accuracy = model.evaluate(test_generator)

# Affichage de la perte et de l'accuracy sur le jeu de test
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Création d'une figure pour visualiser l'évolution de l'entraînement
plt.figure(figsize=(12, 4))

# Premier sous-graphe pour l'accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy over Epochs')

# Deuxième sous-graphe pour la loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss over Epochs')

# Affichage des graphiques
plt.show()

# Génération des prédictions pour les métriques détaillées
print("\nGenerating predictions for evaluation...")

# Réinitialisation du générateur de test pour garantir l'ordre correct
test_generator.reset()

# Prédiction des probabilités sur le jeu de test
predictions = model.predict(test_generator)

# Conversion des probabilités en classes binaires avec un seuil de 0.5
predicted_classes = (predictions > 0.5).astype(int).flatten()

# Récupération des vraies classes
true_classes = test_generator.classes

# Affichage du rapport de classification
print("\nClassification Report:")

# Récupération des noms des classes dans le bon ordre
target_names = [name for name, idx in sorted(train_generator.class_indices.items(), key=lambda item: item[1])]

# Affichage des métriques precision, recall et f1-score
print(classification_report(true_classes, predicted_classes, target_names=target_names))

# Calcul de la matrice de confusion
cm = confusion_matrix(true_classes, predicted_classes)

# Affichage de la matrice de confusion
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# 3-Model Classification des images par VGG16 et SVM

In [ ]:
import os  # Gestion fichiers
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'  # Logs TensorFlow complets

import tensorflow as tf  # Deep learning
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Chargement images
from tensorflow.keras import layers, models  # Couches réseau
from tensorflow.keras.applications import VGG16  # Modèle pré-entraîné
from tensorflow.keras.applications.vgg16 import preprocess_input  # Prétraitement VGG16
import matplotlib.pyplot as plt  # Graphiques
import numpy as np  # Calculs
from sklearn.metrics import classification_report, confusion_matrix  # Évaluation
from sklearn.svm import SVC  # Classifieur SVM
from sklearn.preprocessing import StandardScaler  # Normalisation
import seaborn as sns  # Visualisation

train_data_dir = "Dataset_E_CONT"  # Données entraînement
test_data_dir = "data_test_CONT"  # Données test
weights_path = "models/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"  # Poids VGG16

IMG_HEIGHT = 224; IMG_WIDTH = 224  # Taille images
BATCH_SIZE = 32; NUM_CLASSES = 2  # Paramètres

if not os.path.exists(weights_path): raise FileNotFoundError(f"Poids introuvable : {weights_path}")  # Vérification

train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input, rotation_range=20, width_shift_range=0.2,
                                   height_shift_range=0.2, shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')  # Augmentation

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)  # Test sans augmentation

print("Chargement des données d'entraînement...")  # Info utilisateur
train_generator = train_datagen.flow_from_directory(train_data_dir, target_size=(IMG_HEIGHT, IMG_WIDTH),
                                                    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)  # Train

print("Chargement des données de test...")  # Info utilisateur
test_generator = test_datagen.flow_from_directory(test_data_dir, target_size=(IMG_HEIGHT, IMG_WIDTH),
                                                  batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)  # Test

print("Construction de VGG16 (extracteur de caractéristiques)...")  # Info
base_model = VGG16(weights=None, include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))  # Base CNN
base_model.load_weights(weights_path)  # Chargement poids
base_model.trainable = False  # Gel modèle

feature_extractor = models.Sequential([base_model, layers.GlobalAveragePooling2D()])  # Extracteur

print("Extraction des caractéristiques...")  # Info
train_features = feature_extractor.predict(train_generator, steps=len(train_generator), verbose=1)  # Features train
train_labels = train_generator.classes  # Labels train

test_features = feature_extractor.predict(test_generator, steps=len(test_generator), verbose=1)  # Features test
test_labels = test_generator.classes  # Labels test

print("Normalisation des caractéristiques...")  # Info
scaler = StandardScaler()  # Normalisation
train_features_scaled = scaler.fit_transform(train_features)  # Train
test_features_scaled = scaler.transform(test_features)  # Test

print("Entraînement du modèle SVM...")  # Info
svm_model = SVC(class_weight='balanced', kernel='linear', C=1.0, random_state=42, verbose=True)  # SVM
svm_model.fit(train_features_scaled, train_labels)  # Apprentissage

print("Évaluation du modèle SVM...")  # Info
svm_predictions = svm_model.predict(test_features_scaled)  # Prédictions

print("\nRapport de classification (SVM) :")  # Rapport
target_names = [name for name, idx in sorted(train_generator.class_indices.items(), key=lambda item: item[1])]  # Classes
print(classification_report(test_labels, svm_predictions, target_names=target_names))  # Métriques

cm = confusion_matrix(test_labels, svm_predictions)  # Matrice confusion
plt.figure(figsize=(8, 6))  # Figure
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)  # Heatmap
plt.title('Matrice de confusion (SVM)')  # Titre
plt.ylabel('Classe réelle')  # Axe Y
plt.xlabel('Classe prédite')  # Axe X
plt.show()  # Affichage 


---
# comprendre les décision du modèle utiliser et pourquoi il le prend technique XAI utiliser(Grad-CAM).
---

# Grad-CAM pour comprendre les prediction du model en affichant le heatmap de l'image predit

### - Rouge / jaune : zones importantes pour la décision<br><br> - Bleu / vert : zones moins importantes

### Dans notre cas : si la heatmap est sur les poumons c'est bon signe pour la prediction et si elle est sur les bords, le fond ou du texte ce que le modèle apprend mal




In [ ]:
import matplotlib.pyplot as plt  # Bibliothèque pour afficher les images et graphiques
import matplotlib.cm as cm  # Module pour appliquer une palette de couleurs à la heatmap
import numpy as np  # Bibliothèque pour les calculs numériques et la manipulation des tableaux
import tensorflow as tf  # Framework principal pour le deep learning
from tensorflow.keras.preprocessing import image  # Outils Keras pour charger et convertir les images
from tensorflow.keras.models import load_model  # Fonction pour charger un modèle déjà entraîné
import os  # Module pour manipuler les chemins et noms de fichiers

model_path = "/content/drive/MyDrive/Mes_Projet/Projet/model_colab/meilleur_modele_pneumonia_cnn_cropkg.keras"  # Chemin du modèle CNN sauvegardé
last_conv_layer_name = "conv2d_11"  # Nom de la dernière couche convolutionnelle utilisée pour Grad-CAM
sample_img_path = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E/PNEUMONIA/person1000_bacteria_2931.jpeg"  # Chemin de l’image à analyser
#sample_img_path = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E/NORMAL/IM-0127-0001.jpeg"  # Chemin de l’image à analyser

model = load_model(model_path)  # Chargement du modèle entraîné depuis le fichier .keras
print("Modèle chargé avec succès.")  # Confirmation que le chargement s’est bien passé
_ = model.predict(tf.zeros((1, 224, 224, 3)))  # Prédiction fictive pour forcer la construction complète du modèle

last_conv_layer = model.get_layer(last_conv_layer_name)  # Récupération de la couche convolutionnelle cible par son nom
print(f"Couche convolutionnelle utilisée : {last_conv_layer.name}")  # Vérification de la couche réellement utilisée

def make_gradcam_heatmap(img_array, model, last_conv_layer):  # Fonction qui calcule la carte d’activation Grad-CAM
    with tf.GradientTape() as tape:  # Contexte permettant de calculer les gradients automatiquement
        input_tensor = tf.convert_to_tensor(img_array)  # Conversion de l’image en tenseur TensorFlow
        tape.watch(input_tensor)  # Demande à TensorFlow de suivre les opérations liées à cette entrée

        last_conv_output_model = tf.keras.models.Model(inputs=model.inputs, outputs=last_conv_layer.output)  # Modèle intermédiaire qui retourne les activations de la dernière couche conv
        conv_output = last_conv_output_model(input_tensor)  # Extraction des cartes de caractéristiques de cette couche

        classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])  # Création d’une nouvelle entrée de même forme que la sortie conv
        x = classifier_input  # Initialisation du flux de calcul du classifieur

        last_idx = [i for i, l in enumerate(model.layers) if l.name == last_conv_layer.name][0]  # Recherche de l’index exact de la couche conv dans le modèle

        for layer in model.layers[last_idx + 1:]: x = layer(x)  # Reconstruction de la partie classification après la couche conv

        classifier_model = tf.keras.models.Model(classifier_input, x)  # Création d’un modèle qui va des features conv jusqu’à la prédiction finale
        predictions = classifier_model(conv_output)  # Calcul de la prédiction finale à partir des cartes de caractéristiques

        class_score = predictions[0][0] if predictions[0][0] >= 0.5 else 1 - predictions[0][0]  # Sélection du score de la classe prédite pour guider Grad-CAM

    grads = tape.gradient(class_score, conv_output)  # Calcul des gradients du score par rapport aux activations conv

    if grads is None: return np.zeros(conv_output.shape[1:3])  # Retour d’une heatmap vide si le gradient n’a pas pu être calculé

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # Moyenne des gradients sur la hauteur, largeur et batch pour obtenir l’importance de chaque canal

    conv_output = conv_output[0]  # Sélection de l’unique image du batch
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]  # Pondération des cartes de caractéristiques par leur importance
    heatmap = tf.squeeze(heatmap)  # Suppression des dimensions inutiles pour obtenir une carte 2D

    heatmap = np.maximum(heatmap, 0)  # Suppression des valeurs négatives pour ne garder que les contributions positives
    max_val = tf.reduce_max(heatmap)  # Recherche de la valeur maximale de la heatmap

    heatmap = heatmap * 0 if max_val == 0 else heatmap / max_val  # Normalisation de la heatmap entre 0 et 1

    return heatmap.numpy()  # Conversion finale de la heatmap en tableau NumPy

def display_gradcam(img_path, heatmap, model, img_array, alpha=0.4):  # Fonction qui superpose la heatmap sur l’image originale
    img_display = image.img_to_array(image.load_img(img_path))  # Chargement de l’image originale en tableau numérique

    heatmap_rescaled = np.uint8(255 * heatmap)  # Conversion de la heatmap normalisée en valeurs entre 0 et 255

    jet = cm.get_cmap("jet")  # Choix de la palette de couleurs "jet" pour mieux visualiser les zones importantes
    jet_colors = jet(np.arange(256))[:, :3]  # Extraction des composantes RGB de la palette
    jet_heatmap = image.array_to_img(jet_colors[heatmap_rescaled])  # Transformation de la heatmap en image couleur
    jet_heatmap = image.img_to_array(jet_heatmap.resize((img_display.shape[1], img_display.shape[0])))  # Redimensionnement à la taille de l’image originale

    superimposed_img = image.array_to_img((jet_heatmap * alpha + img_display).astype(np.uint8))  # Fusion de la heatmap et de l’image originale avec transparence

    target_names = ['NORMAL', 'PNEUMONIA']  # Noms des deux classes du problème de classification

    pred = model.predict(img_array)[0][0]  # Récupération de la probabilité prédite par le modèle
    pred_idx = (pred >= 0.5).astype(int)  # Conversion de la probabilité en classe binaire selon le seuil 0.5
    pred_name = target_names[pred_idx]  # Récupération du nom de la classe prédite

    pred_text = f"Prédiction : {pred_name} (Prob : {pred if pred_idx else 1-pred:.2f})"  # Création d’un texte lisible avec la classe prédite et sa probabilité

    actual_class = os.path.basename(os.path.dirname(img_path)).upper()  # Extraction automatique de la classe réelle à partir du dossier parent de l’image

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))  # Création d’une figure avec deux sous-images côte à côte

    axes[0].imshow(superimposed_img); axes[0].axis('off'); axes[0].set_title('Visualisation Grad-CAM')  # Affichage de l’image avec heatmap superposée
    axes[1].imshow(img_display.astype(np.uint8)); axes[1].axis('off'); axes[1].set_title(f"{pred_text}\nClasse réelle : {actual_class}")  # Affichage de l’image originale avec résultat de prédiction

    plt.tight_layout(); plt.show()  # Ajustement de l’affichage puis rendu final à l’écran

print(f"\nTraitement de l'image : {sample_img_path}")  # Message indiquant l’image actuellement analysée

img = image.load_img(sample_img_path, target_size=(224, 224))  # Chargement de l’image en la redimensionnant au format attendu par le modèle
img_array = image.img_to_array(img)  # Conversion de l’image en tableau NumPy
img_array = np.expand_dims(img_array, axis=0) / 255.0  # Ajout de la dimension batch et normalisation des pixels entre 0 et 1

heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer)  # Génération de la carte Grad-CAM à partir de l’image
display_gradcam(sample_img_path, heatmap, model, img_array)  # Affichage final de la visualisation explicative